# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Each OOG container must be assigned to exactly one vessel slot
- Each slot can accommodate at most one container
- Weight and height constraints must be respected for each assignment
- Bay weight limits must be satisfied to maintain vessel stability
- Assignment costs, stability factors, and lashing requirements are known

### Approach (step-by-step)
1. **Problem Formulation**: Define the multi-objective assignment problem with decision variables
2. **Objective Functions**: Minimize cost, maximize stability, minimize lashing complexity
3. **Constraints**: Ensure feasibility with weight, height, and capacity limitations
4. **Solution Method**: Use Hungarian algorithm for the cost minimization objective
5. **Analysis**: Evaluate solution quality and constraint satisfaction

### What to look for in the results
- Optimal assignment that minimizes total cost while satisfying all constraints
- Container-slot assignments with clear justification
- Total cost calculation and capacity utilization analysis
- Constraint verification for weight, height, and bay limits

### Concrete example (from the source)
We'll solve the OOG container placement problem with:
- 3 OOG containers with different weights and heights
- 4 available slots with varying capacities
- Assignment cost matrix for all container-slot combinations
- Hungarian algorithm to find the optimal cost-minimizing assignment

In [ ]:
# Import required libraries for mathematical optimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set up professional visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries imported successfully for OOG container placement optimization")

In [ ]:
@dataclass
class OOGContainer:
    """Represents an Out-of-Gauge container with specific characteristics"""
    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str  # flat rack, open-top, platform
    lashing_coeff: float  # lashing requirement coefficient

@dataclass
class VesselSlot:
    """Represents an available vessel slot with constraints"""
    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int  # bay identifier for stability calculations
    crane_distance: float  # distance to nearest crane

@dataclass
class AssignmentProblem:
    """Contains the complete OOG container assignment problem"""
    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray  # assignment costs
    stability_matrix: np.ndarray  # stability factors
    bay_weight_limits: Dict[int, float]  # maximum weight per bay

print("✅ Data structures defined for OOG container placement")

In [ ]:
# Define the concrete example from the source material
# OOG containers with their characteristics
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9)
]

# Available vessel slots with constraints
slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0),
    VesselSlot(2, 40, 5.0, 3, 12.0),
    VesselSlot(3, 45, 3.0, 3, 18.0),
    VesselSlot(4, 60, 4.0, 3, 10.0)
]

# Assignment cost matrix from the source (rows: containers, columns: slots)
cost_matrix = np.array([
    [100, 150, 120, 90],   # Container 1 costs
    [80, 70, 110, 85],     # Container 2 costs
    [95, 140, 100, 80]     # Container 3 costs
])

# Stability factors (higher is better)
stability_matrix = np.array([
    [0.8, 0.7, 0.9, 0.85],
    [0.75, 0.8, 0.7, 0.9],
    [0.85, 0.65, 0.8, 0.75]
])

# Bay weight limits for vessel stability
bay_weight_limits = {2: 80, 3: 120}  # tons per bay

# Create the complete problem instance
problem = AssignmentProblem(containers, slots, cost_matrix, stability_matrix, bay_weight_limits)

print(f"📊 Problem defined: {len(containers)} containers, {len(slots)} slots")
print(f"📋 Cost matrix shape: {cost_matrix.shape}")
print(f"⚖️  Bay weight limits: {bay_weight_limits}")

In [ ]:
def solve_hungarian_algorithm(cost_matrix: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Solve the assignment problem using Hungarian algorithm
    
    Args:
        cost_matrix: Cost matrix for container-slot assignments
        
    Returns:
        Tuple of (assignment_array, total_cost)
    """
    # Use scipy's implementation of the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Create assignment matrix
    n_containers, n_slots = cost_matrix.shape
    assignment = np.zeros((n_containers, n_slots), dtype=int)
    
    # Fill assignment matrix
    for i, j in zip(row_ind, col_ind):
        assignment[i, j] = 1
    
    # Calculate total cost
    total_cost = cost_matrix[row_ind, col_ind].sum()
    
    return assignment, total_cost

def verify_constraints(problem: AssignmentProblem, assignment: np.ndarray) -> Dict[str, bool]:
    """
    Verify all constraints for the given assignment
    
    Args:
        problem: Complete problem instance
        assignment: Assignment matrix
        
    Returns:
        Dictionary of constraint satisfaction results
    """
    constraints = {}
    
    # Check 1: Each container assigned exactly once
    container_assignments = assignment.sum(axis=1)
    constraints['each_container_assigned'] = np.all(container_assignments == 1)
    
    # Check 2: Each slot used at most once
    slot_assignments = assignment.sum(axis=0)
    constraints['each_slot_max_once'] = np.all(slot_assignments <= 1)
    
    # Check 3: Weight capacity constraints
    constraints['weight_capacity'] = True
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                if container.weight > slot.weight_capacity:
                    constraints['weight_capacity'] = False
                    break
    
    # Check 4: Height constraints
    constraints['height_constraints'] = True
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                if container.height > slot.height_limit:
                    constraints['height_constraints'] = False
                    break
    
    # Check 5: Bay weight limits
    constraints['bay_weight_limits'] = True
    bay_weights = {}
    for bay_id in problem.bay_weight_limits:
        bay_weights[bay_id] = 0
    
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                bay_weights[slot.bay_id] += container.weight
    
    for bay_id, total_weight in bay_weights.items():
        if total_weight > problem.bay_weight_limits[bay_id]:
            constraints['bay_weight_limits'] = False
            break
    
    return constraints

print("✅ Optimization and constraint verification functions defined")

In [ ]:
# Solve the assignment problem using Hungarian algorithm
assignment, total_cost = solve_hungarian_algorithm(problem.cost_matrix)

print("🔍 SOLVING OOG CONTAINER ASSIGNMENT PROBLEM")
print("=" * 50)
print(f"💰 Total assignment cost: {total_cost} units")
print()

# Display the optimal assignment
print("📋 OPTIMAL ASSIGNMENT:")
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if assignment[i, j] == 1:
            print(f"  Container {container.id} ({container.container_type}, {container.weight}t, {container.height}m) → Slot {slot.id} (capacity: {slot.weight_capacity}t, height: {slot.height_limit}m, bay: {slot.bay_id})")
            print(f"    Cost: {problem.cost_matrix[i, j]}, Stability: {problem.stability_matrix[i, j]:.2f}")
print()

# Verify all constraints
constraints = verify_constraints(problem, assignment)
print("⚖️  CONSTRAINT VERIFICATION:")
all_satisfied = True
for constraint, satisfied in constraints.items():
    status = "✅" if satisfied else "❌"
    print(f"  {status} {constraint.replace('_', ' ').title()}: {satisfied}")
    if not satisfied:
        all_satisfied = False

print(f"\n🎯 Overall feasibility: {'✅ All constraints satisfied' if all_satisfied else '❌ Constraints violated'}")

In [ ]:
def calculate_performance_metrics(problem: AssignmentProblem, assignment: np.ndarray) -> Dict[str, float]:
    """
    Calculate detailed performance metrics for the assignment
    
    Args:
        problem: Complete problem instance
        assignment: Assignment matrix
        
    Returns:
        Dictionary of performance metrics
    """
    metrics = {}
    
    # Total cost (already calculated)
    metrics['total_cost'] = problem.cost_matrix[assignment == 1].sum()
    
    # Total stability score
    metrics['total_stability'] = problem.stability_matrix[assignment == 1].sum()
    
    # Average capacity utilization
    total_capacity_used = 0
    total_capacity_available = 0
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                total_capacity_used += container.weight
                total_capacity_available += slot.weight_capacity
    
    metrics['capacity_utilization'] = (total_capacity_used / total_capacity_available) * 100
    
    # Lashing complexity score
    total_lashing_score = 0
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                total_lashing_score += container.lashing_coeff * slot.crane_distance
    
    metrics['lashing_complexity'] = total_lashing_score
    
    # Bay weight distribution
    bay_weights = {}
    for bay_id in problem.bay_weight_limits:
        bay_weights[bay_id] = 0
    
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                bay_weights[slot.bay_id] += container.weight
    
    metrics['bay_weights'] = bay_weights
    
    return metrics

# Calculate performance metrics
metrics = calculate_performance_metrics(problem, assignment)

print("📊 PERFORMANCE ANALYSIS")
print("=" * 30)
print(f"💰 Total Cost: {metrics['total_cost']} units")
print(f"⚖️  Total Stability Score: {metrics['total_stability']:.2f}")
print(f"📦 Capacity Utilization: {metrics['capacity_utilization']:.1f}%")
print(f"🔗 Lashing Complexity: {metrics['lashing_complexity']:.1f}")
print()
print("⚓ Bay Weight Distribution:")
for bay_id, weight in metrics['bay_weights'].items():
    limit = problem.bay_weight_limits[bay_id]
    utilization = (weight / limit) * 100
    print(f"  Bay {bay_id}: {weight}t / {limit}t ({utilization:.1f}% utilization)")

In [ ]:
# Create comprehensive visualization of the assignment solution
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('OOG Container Assignment Analysis - Mathematical Formulation', fontsize=16, fontweight='bold')

# 1. Cost matrix heatmap with optimal assignment
sns.heatmap(cost_matrix, annot=True, fmt='d', cmap='RdYlBu_r', ax=ax1,
           xticklabels=[f'Slot {s.id}' for s in slots],
           yticklabels=[f'Container {c.id}' for c in containers])
ax1.set_title('Assignment Cost Matrix')
ax1.set_xlabel('Vessel Slots')
ax1.set_ylabel('OOG Containers')

# Highlight optimal assignments
for i, container in enumerate(containers):
    for j, slot in enumerate(slots):
        if assignment[i, j] == 1:
            ax1.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='green', linewidth=3))

# 2. Container characteristics comparison
container_names = [f"C{c.id}\n({c.container_type})" for c in containers]
weights = [c.weight for c in containers]
heights = [c.height for c in containers]

x_pos = np.arange(len(container_names))
width = 0.35

ax2.bar(x_pos - width/2, weights, width, label='Weight (tons)', alpha=0.8, color='skyblue')
ax2_twin = ax2.twinx()
ax2_twin.bar(x_pos + width/2, heights, width, label='Height (m)', alpha=0.8, color='lightcoral')

ax2.set_xlabel('Containers')
ax2.set_ylabel('Weight (tons)', color='skyblue')
ax2_twin.set_ylabel('Height (m)', color='lightcoral')
ax2.set_title('Container Characteristics')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(container_names)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

# 3. Slot capacity utilization
slot_names = [f"Slot {s.id}\n(Bay {s.bay_id})" for s in slots]
capacities = [s.weight_capacity for s in slots]
used_capacity = [0] * len(slots)

for i, container in enumerate(containers):
    for j, slot in enumerate(slots):
        if assignment[i, j] == 1:
            used_capacity[j] = container.weight

x_pos = np.arange(len(slot_names))
ax3.bar(x_pos, capacities, alpha=0.3, label='Available Capacity', color='gray')
ax3.bar(x_pos, used_capacity, alpha=0.8, label='Used Capacity', color='orange')

ax3.set_xlabel('Vessel Slots')
ax3.set_ylabel('Weight (tons)')
ax3.set_title('Slot Capacity Utilization')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(slot_names)
ax3.legend()

# Add utilization percentages
for i, (used, total) in enumerate(zip(used_capacity, capacities)):
    if used > 0:
        utilization = (used / total) * 100
        ax3.text(i, used + 1, f'{utilization:.1f}%', ha='center', va='bottom', fontweight='bold')

# 4. Bay weight distribution
bay_ids = list(problem.bay_weight_limits.keys())
bay_limits = [problem.bay_weight_limits[bid] for bid in bay_ids]
bay_weights_actual = [metrics['bay_weights'][bid] for bid in bay_ids]

x_pos = np.arange(len(bay_ids))
width = 0.35

ax4.bar(x_pos - width/2, bay_limits, width, label='Weight Limit', alpha=0.3, color='red')
ax4.bar(x_pos + width/2, bay_weights_actual, width, label='Actual Weight', alpha=0.8, color='green')

ax4.set_xlabel('Bay ID')
ax4.set_ylabel('Weight (tons)')
ax4.set_title('Bay Weight Distribution')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f'Bay {bid}' for bid in bay_ids])
ax4.legend()

# Add utilization percentages for bays
for i, (actual, limit) in enumerate(zip(bay_weights_actual, bay_limits)):
    utilization = (actual / limit) * 100
    ax4.text(i, actual + 1, f'{utilization:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 Comprehensive visualization created for OOG container assignment analysis")

In [ ]:
# Sensitivity analysis - What if scenarios
print("🔍 SENSITIVITY ANALYSIS - WHAT IF SCENARIOS")
print("=" * 55)

# Scenario 1: Increased cost for Slot 4 (originally optimal)
modified_cost_matrix_1 = cost_matrix.copy()
modified_cost_matrix_1[:, 3] += 50  # Increase costs for Slot 4

assignment_1, cost_1 = solve_hungarian_algorithm(modified_cost_matrix_1)
print(f"\n📈 Scenario 1: Slot 4 costs increased by 50 units")
print(f"   New total cost: {cost_1} units (was {total_cost})")
print(f"   Cost increase: {cost_1 - total_cost} units ({((cost_1/total_cost - 1)*100):.1f}%)")

# Show new assignment
for i, container in enumerate(containers):
    for j, slot in enumerate(slots):
        if assignment_1[i, j] == 1:
            print(f"   Container {container.id} → Slot {slot.id} (cost: {modified_cost_matrix_1[i, j]})")

# Scenario 2: Reduced capacity in Slot 2
print(f"\n📉 Scenario 2: Slot 2 capacity reduced from 40t to 30t")
modified_slots = slots.copy()
modified_slots[1] = VesselSlot(2, 30, 5.0, 3, 12.0)  # Reduced capacity

# Check if original assignment is still feasible
original_feasible = True
for i, container in enumerate(containers):
    for j, slot in enumerate(modified_slots):
        if assignment[i, j] == 1:
            if container.weight > slot.weight_capacity:
                original_feasible = False
                print(f"   ❌ Original assignment infeasible: Container {container.id} ({container.weight}t) exceeds Slot {slot.id} capacity ({slot.weight_capacity}t)")
                break

if original_feasible:
    print(f"   ✅ Original assignment still feasible")

# Scenario 3: Different objective - Maximize stability instead of minimize cost
print(f"\n⚖️  Scenario 3: Maximize stability instead of minimize cost")
# Convert stability to cost (higher stability = lower "cost")
stability_cost_matrix = (1 - stability_matrix) * 100  # Invert and scale

assignment_3, stability_cost = solve_hungarian_algorithm(stability_cost_matrix)
actual_stability = stability_matrix[assignment_3 == 1].sum()
actual_cost = cost_matrix[assignment_3 == 1].sum()

print(f"   Stability-optimized total stability: {actual_stability:.2f} (was {metrics['total_stability']:.2f})")
print(f"   Stability-optimized total cost: {actual_cost} units (was {total_cost})")
print(f"   Trade-off: +{actual_cost - total_cost} cost for +{actual_stability - metrics['total_stability']:.2f} stability")

# Show stability-optimized assignment
for i, container in enumerate(containers):
    for j, slot in enumerate(slots):
        if assignment_3[i, j] == 1:
            print(f"   Container {container.id} → Slot {slot.id} (stability: {stability_matrix[i, j]:.2f}, cost: {cost_matrix[i, j]})")

print(f"\n🎯 Key Insights:")
print(f"   • Cost optimization is sensitive to slot pricing changes")
print(f"   • Capacity reductions can make optimal assignments infeasible")
print(f"   • Different objectives (cost vs stability) lead to different assignments")
print(f"   • Trade-offs between objectives must be considered in practice")

### Why this Tier exists vs earlier Tiers

**Tier 1: Mathematical Formulation** provides the foundational approach with several key advantages:

**Advantages vs other approaches:**
- **Optimality Guarantee**: The Hungarian algorithm provides a provably optimal solution for the assignment problem
- **Transparent Logic**: Mathematical formulation makes the problem structure and constraints explicit
- **Fast Execution**: Polynomial time complexity (O(n³)) makes it suitable for real-time decision making
- **Reproducible Results**: Deterministic algorithm produces consistent results
- **Constraint Verification**: Systematic checking of all operational constraints

**Disadvantages:**
- **Single Objective**: Optimizes only one objective (cost) at a time
- **Limited Flexibility**: Cannot easily handle complex, non-linear constraints
- **Static Parameters**: Assumes all parameters are known and constant
- **No Learning**: Cannot adapt to new patterns or historical data

### When to use this Tier

**Use Mathematical Formulation when:**
- Problem size is moderate (up to hundreds of containers/slots)
- All parameters and constraints are well-defined
- Optimal solution is required for critical operations
- Single objective optimization is sufficient
- Fast execution time is essential
- Regulatory compliance requires provable optimality

**This Tier forms the foundation** for understanding the OOG container placement problem and provides a benchmark against which more advanced methods (heuristics, metaheuristics, AI/ML) can be compared.